# Notebook 3 — Validate sequence-level statistics (fig5–fig7 territory)

**Prerequisites.**
- Notebook 1: the C sieve produces the correct histogram.
- Notebook 2: the Hardy–Littlewood / Wolf theory layer is correctly implemented (the large `emp/Wolf` ratios at $g=6,10,12$ are physics, not a bug).

**What's left to check.** The original `summary.md` also reports three sequence-level results:

1. **Jumping-champion transitions** (fig5). Claimed: `2 → 4 → 2 → 6 → 2 → 6 → 2 → 4 → 6` at specific $x$ values, final stabilisation at $g=6$ around $x \approx 568$.
2. **Running minimum $g_n / \ln p_n$** (fig6). Claimed: minimum $\approx 0.1086$ near $p \approx 10^{8}$ (so some gap $g$ with $g / \ln p \approx 0.11$, i.e. $g \approx 0.11 \cdot 18.4 \approx 2$, which must be the twin at some large prime).
3. **Autocorrelation $\rho(k)$ of consecutive gaps** (fig7). Claimed: $\rho(1) = -0.0316$, $z = -78.65$ (huge antisignal). This is the Oliver–Soundararajan-style structural deviation from Cramér independence.

These were produced by a local `.py` that we do not trust. We rebuild each statistic from the numpy prime stream (no C CSV, no buggy `.py`) and compare numerically. Key targets:

- **Jumping-champion pipeline.** Naive implementations get the transition $x$ wrong by tie-breaking (multiple $g$ sharing the same count) or by using a log-grid that skips past a transition. We use a dense grid *and* an event-driven scan (record every step where $\text{argmax}$ changes) and check both agree.
- **Autocorrelation.** The reported $\rho(1) = -0.032$ at $z = -78$ is extreme. We compute it three ways: biased estimator, unbiased estimator, and via a random-shuffle null with $N_{\text{shuffles}} = 100$. All three must give consistent answers. If the $z$ is genuine, the structural antisignal is real (Oliver–Soundararajan consequence). If any of the three disagrees, the published number is a pipeline artefact.

**Notation.** $g_n = p_{n+1} - p_n$, indexed from $n = 1$ with $p_1 = 2$. The sequence starts `g = [1, 2, 2, 4, 2, 4, 2, 4, 6, ...]`. The first entry (gap=1, pair 2→3) is the only odd gap — we drop it for autocorrelation and champion tracking, since it's a one-off artefact.

In [1]:
import sys, time, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print('python', sys.version.split()[0], '| numpy', np.__version__,
      '| pandas', pd.__version__)

N_LIMIT = 10**8

def sieve_primes(n: int) -> np.ndarray:
    if n < 2: return np.array([], dtype=np.int64)
    is_prime = np.ones(n + 1, dtype=bool)
    is_prime[:2] = False
    is_prime[4::2] = False
    for i in range(3, int(math.isqrt(n)) + 1, 2):
        if is_prime[i]:
            is_prime[i*i::2*i] = False
    return np.flatnonzero(is_prime).astype(np.int64)

t0 = time.perf_counter()
primes = sieve_primes(N_LIMIT)
gaps_all = np.diff(primes).astype(np.int64)    # includes the first (gap=1)
p_start_all = primes[:-1]                       # smaller prime of each gap
# Drop the singular gap=1 (pair 2,3) for all sequence-level stats.
mask = gaps_all >= 2
gaps = gaps_all[mask]
p_start = p_start_all[mask]
print(f'sieve done in {time.perf_counter()-t0:.2f}s | '
      f'pi(N) = {len(primes):,} | gaps used = {len(gaps):,}')

python 3.11.10 | numpy 2.0.2 | pandas 2.2.3
sieve done in 0.44s | pi(N) = 5,761,455 | gaps used = 5,761,453


## 1. Jumping-champion transitions

Definition: for each $x \ge 2$ let $H_x$ be the histogram of gaps $g_n$ with $p_n \le x$. The *champion* at $x$ is $\arg\max_g H_x[g]$, with ties broken by the smallest $g$.

Naive approach: walk $p_n$ in order; after every insertion, check if the current $\arg\max$ changed. This is $O(P)$ histogram updates and $O(P \cdot g_{\max})$ if we scan argmax every step, but we only need to update argmax when the bin we just incremented could overtake the leader — cheap.

We implement the **event-driven** version and report *every* transition (no grid).

In [2]:
def champion_transitions(gaps: np.ndarray, p_start: np.ndarray):
    max_g = int(gaps.max())
    counts = np.zeros(max_g + 1, dtype=np.int64)
    current_champ = -1
    current_count = -1
    transitions = []   # list of (x, old_champ, new_champ, new_count)
    for i, (g, p) in enumerate(zip(gaps, p_start)):
        counts[g] += 1
        # New candidate leader logic:
        # - counts[g] overtakes current leader strictly -> new champ = g
        # - if counts[g] == current_count and g < current_champ -> new champ = g
        if g == current_champ:
            current_count = counts[g]
        else:
            if counts[g] > current_count or (
                counts[g] == current_count and g < current_champ):
                transitions.append((int(p), current_champ, int(g), int(counts[g])))
                current_champ, current_count = int(g), int(counts[g])
    return transitions

t0 = time.perf_counter()
trans = champion_transitions(gaps, p_start)
print(f'event-driven scan done in {time.perf_counter()-t0:.1f}s')
print(f'total transitions: {len(trans)}')
print()
print(f'{"p":>12} | {"old":>4} | {"new":>4} | count_at_event')
for p, old, new, c in trans[:40]:
    print(f'{p:>12,} | {old:>4} | {new:>4} | {c}')

event-driven scan done in 2.7s
total transitions: 11

           p |  old |  new | count_at_event
           3 |   -1 |    2 | 1
         127 |    2 |    4 | 11
         137 |    4 |    2 | 11
         383 |    2 |    6 | 22
         419 |    6 |    2 | 22
         443 |    2 |    6 | 24
         461 |    6 |    2 | 24
         487 |    2 |    4 | 25
         557 |    4 |    6 | 27
         937 |    6 |    4 | 39
         941 |    4 |    6 | 40


In [3]:
# Compare with the claimed list from summary.md (fig5).
# The original used a log-grid of 400 points, so its `x` is not exact — it is
# the grid point at which the change was FIRST OBSERVED, i.e. an upper bound on
# the true transition p.  For every claimed (x, new_champ), we must find a real
# transition with new_champ matching and p <= x.
claimed = [
    (1.000e+01, 2),
    (1.274e+02, 4),
    (1.381e+02, 2),
    (3.949e+02, 6),
    (4.281e+02, 2),
    (4.458e+02, 6),
    (4.642e+02, 2),
    (5.032e+02, 4),
    (5.680e+02, 6),
]
print(f'{"claim_x":>10} | {"claim_new":>9} | matching real transition (p, old→new)')
for cx, cnew in claimed:
    # nearest earlier true transition into `cnew`
    hits = [(p, old, new) for (p, old, new, _) in trans
            if new == cnew and p <= cx]
    if hits:
        p, old, new = hits[-1]
        print(f'{cx:>10.1f} | {cnew:>9} | p={p}, {old}→{new}')
    else:
        print(f'{cx:>10.1f} | {cnew:>9} | NO MATCH found ≤ {cx}')

print('\n(Any "NO MATCH" means the fig5 list is wrong at that row.)')

   claim_x | claim_new | matching real transition (p, old→new)
      10.0 |         2 | p=3, -1→2
     127.4 |         4 | p=127, 2→4
     138.1 |         2 | p=137, 4→2
     394.9 |         6 | p=383, 2→6
     428.1 |         2 | p=419, 6→2
     445.8 |         6 | p=443, 2→6
     464.2 |         2 | p=461, 6→2
     503.2 |         4 | p=487, 2→4
     568.0 |         6 | p=557, 4→6

(Any "NO MATCH" means the fig5 list is wrong at that row.)


## 2. Running minimum $g_n / \ln p_n$

Straightforward: compute ratio per gap, take `np.minimum.accumulate`. Report the minimum and the prime where it occurs.

In [4]:
ratio = gaps / np.log(p_start)
running = np.minimum.accumulate(ratio)
imin = int(running.argmin())
print(f'minimum g/ln p      : {running[imin]:.6f}')
print(f'attained at p_start : {int(p_start[imin]):,}')
print(f'corresponding gap g : {int(gaps[imin])}')
print(f'summary.md claim    : 0.1086 at p ≈ 1.000e+08')
print(f'match?              : {abs(running[imin] - 0.1086) < 1e-3}')

# Also list the ten records (each time a new minimum is set) for reference.
is_new_min = np.r_[True, ratio[1:] < running[:-1]]
new_min_idx = np.where(is_new_min)[0]
print(f'\nnumber of new-minimum events: {len(new_min_idx)}')
print('last 10 record-low ratios:')
print(f'{"p_start":>12} | {"gap":>4} | {"ratio":>9} | ln_p')
for i in new_min_idx[-10:]:
    print(f'{int(p_start[i]):>12,} | {int(gaps[i]):>4} | '
          f'{ratio[i]:>9.6f} | {math.log(p_start[i]):.4f}')

minimum g/ln p      : 0.108574
attained at p_start : 99,999,587
corresponding gap g : 2
summary.md claim    : 0.1086 at p ≈ 1.000e+08
match?              : True

number of new-minimum events: 440312
last 10 record-low ratios:
     p_start |  gap |     ratio | ln_p
  99,997,187 |    2 |  0.108574 | 18.4207
  99,997,367 |    2 |  0.108574 | 18.4207
  99,997,871 |    2 |  0.108574 | 18.4207
  99,998,447 |    2 |  0.108574 | 18.4207
  99,998,609 |    2 |  0.108574 | 18.4207
  99,999,077 |    2 |  0.108574 | 18.4207
  99,999,257 |    2 |  0.108574 | 18.4207
  99,999,437 |    2 |  0.108574 | 18.4207
  99,999,539 |    2 |  0.108574 | 18.4207
  99,999,587 |    2 |  0.108574 | 18.4207


## 3. Autocorrelation of consecutive gaps

This is the highest-stakes check. The claimed $\rho(1) = -0.0316$ at $z = -78.65$ is a very strong statement (structural dependence between consecutive prime gaps). We redo it with three independent computations:

- **biased** $\rho(k) = \frac{1}{(N) \sigma^2} \sum_{i=1}^{N-k} (g_i - \bar g)(g_{i+k} - \bar g)$
- **unbiased** $\rho(k) = \frac{1}{(N-k) \sigma^2} \sum \ldots$
- **shuffle null**: recompute $\rho(k)$ on random permutations of the same gap multiset to get the standard deviation $\sigma_{\text{shuf}}(k)$ under the null hypothesis of independence; $z(k) = \rho(k) / \sigma_{\text{shuf}}(k)$.

In [5]:
g = gaps.astype(np.float64)
g -= g.mean()
var = (g * g).mean()
N = len(g)
max_lag = 10

def acorr_biased(x, max_lag):
    N = len(x)
    out = np.empty(max_lag + 1)
    out[0] = 1.0
    for k in range(1, max_lag + 1):
        out[k] = np.mean(x[:-k] * x[k:]) / var
    return out

def acorr_unbiased(x, max_lag):
    N = len(x)
    out = np.empty(max_lag + 1)
    out[0] = 1.0
    for k in range(1, max_lag + 1):
        out[k] = (x[:-k] * x[k:]).sum() / ((N - k) * var)
    return out

rho_b = acorr_biased(g, max_lag)
rho_u = acorr_unbiased(g, max_lag)
print('Biased vs unbiased estimators (should agree to 1e-7 for large N):')
for k in range(1, max_lag + 1):
    print(f'  k={k:2d}  biased={rho_b[k]:+.6f}  unbiased={rho_u[k]:+.6f}  '
          f'diff={rho_b[k]-rho_u[k]:+.2e}')

Biased vs unbiased estimators (should agree to 1e-7 for large N):
  k= 1  biased=-0.031607  unbiased=-0.031607  diff=+6.94e-18
  k= 2  biased=-0.012777  unbiased=-0.012777  diff=+0.00e+00
  k= 3  biased=-0.007325  unbiased=-0.007325  diff=+0.00e+00
  k= 4  biased=-0.004568  unbiased=-0.004568  diff=-8.67e-19
  k= 5  biased=-0.001937  unbiased=-0.001937  diff=-2.17e-19
  k= 6  biased=-0.001606  unbiased=-0.001606  diff=+0.00e+00
  k= 7  biased=-0.000868  unbiased=-0.000868  diff=-1.08e-19
  k= 8  biased=+0.000455  unbiased=+0.000455  diff=-5.42e-20
  k= 9  biased=+0.001050  unbiased=+0.001050  diff=+0.00e+00
  k=10  biased=+0.001382  unbiased=+0.001382  diff=+0.00e+00


In [6]:
# Shuffle null: 100 permutations. Under H_0 (iid), rho(k) has sigma ~ 1/sqrt(N).
# The analytic expectation: sigma_null ≈ 1/sqrt(N) ≈ 1/sqrt(5.76e6) ≈ 4.17e-4.
rng = np.random.default_rng(0)
n_shuffles = 100
null = np.empty((n_shuffles, max_lag + 1))
gs = g.copy()
t0 = time.perf_counter()
for i in range(n_shuffles):
    rng.shuffle(gs)
    null[i] = acorr_biased(gs, max_lag)
print(f'shuffles done in {time.perf_counter()-t0:.1f}s')

sig = null.std(axis=0, ddof=1)
mu  = null.mean(axis=0)
print(f'\n{"k":>3} | {"rho_emp":>10} | {"mu_null":>10} | {"sigma_null":>11} | '
      f'{"z":>7} | {"1/sqrt(N)":>10}')
inv_sqrt_N = 1.0 / math.sqrt(N)
for k in range(1, max_lag + 1):
    z = (rho_b[k] - mu[k]) / sig[k]
    print(f'{k:>3} | {rho_b[k]:>+10.5f} | {mu[k]:>+10.5f} | '
          f'{sig[k]:>11.6f} | {z:>+7.2f} | {inv_sqrt_N:>10.6f}')

print('\nsummary.md claim: rho(1) = -0.0316, z = -78.65')
print(f'our rho(1)      : {rho_b[1]:+.4f}')
print(f'our z(1)        : {(rho_b[1]-mu[1])/sig[1]:+.2f}')

shuffles done in 21.0s

  k |    rho_emp |    mu_null |  sigma_null |       z |  1/sqrt(N)
  1 |   -0.03161 |   -0.00003 |    0.000422 |  -74.77 |   0.000417
  2 |   -0.01278 |   -0.00002 |    0.000451 |  -28.28 |   0.000417
  3 |   -0.00732 |   -0.00002 |    0.000444 |  -16.43 |   0.000417
  4 |   -0.00457 |   +0.00007 |    0.000445 |  -10.42 |   0.000417
  5 |   -0.00194 |   -0.00002 |    0.000442 |   -4.34 |   0.000417
  6 |   -0.00161 |   -0.00004 |    0.000405 |   -3.86 |   0.000417
  7 |   -0.00087 |   +0.00002 |    0.000397 |   -2.23 |   0.000417
  8 |   +0.00046 |   -0.00002 |    0.000417 |   +1.14 |   0.000417
  9 |   +0.00105 |   +0.00001 |    0.000376 |   +2.76 |   0.000417
 10 |   +0.00138 |   +0.00003 |    0.000442 |   +3.07 |   0.000417

summary.md claim: rho(1) = -0.0316, z = -78.65
our rho(1)      : -0.0316
our z(1)        : -74.77


## 4. Autocorrelation — deeper sanity check

If $\rho(1)$ is real, it should survive every reasonable transformation. Two quick tests:

- **Split-half consistency.** Compute $\rho(1)$ on the first half and the second half separately. They must agree to their individual shuffle-sigma.
- **FFT-based full ACF.** `np.correlate` / FFT method for $\rho(1)$ must match the direct sum.

In [7]:
half = N // 2
g1 = gaps[:half].astype(np.float64); g1 -= g1.mean()
g2 = gaps[half:].astype(np.float64); g2 -= g2.mean()
rho1 = (g1[:-1] * g1[1:]).mean() / (g1 * g1).mean()
rho2 = (g2[:-1] * g2[1:]).mean() / (g2 * g2).mean()
print(f'rho(1) first half  : {rho1:+.5f}')
print(f'rho(1) second half : {rho2:+.5f}')
print(f'rho(1) full        : {rho_b[1]:+.5f}')
print(f'sigma_null per half: ~ {1/math.sqrt(half):.5f}')

# FFT-based cross-check
gg = gaps.astype(np.float64) - gaps.mean()
# Zero-pad to next power of two for speed
M = 1 << (2 * len(gg) - 1).bit_length()
F = np.fft.rfft(gg, M)
acf = np.fft.irfft(F * F.conj(), M)[:max_lag + 1]
acf /= acf[0]
print(f'\nrho via FFT, k=1: {acf[1]:+.6f}  (should match biased estimator)')

rho(1) first half  : -0.03342
rho(1) second half : -0.03483
rho(1) full        : -0.03161
sigma_null per half: ~ 0.00059

rho via FFT, k=1: -0.031607  (should match biased estimator)


## 5. Verdict

- [ ] Section 1: every claimed champion transition from fig5 has a matching real transition with $p \le $ claimed $x$.
- [ ] Section 2: running minimum $\approx 0.1086$ at $p \approx 10^{8}$.
- [ ] Section 3: biased and unbiased $\rho(k)$ agree; shuffle $\sigma_k \approx 1/\sqrt{N}$; $\rho(1) \approx -0.032$, $z(1) \approx -78$.
- [ ] Section 4: split-half $\rho(1)$ values agree and are both negative; FFT matches biased estimator.

All green → the sequence-level statistics in `summary.md` are correct and the massive autocorrelation antisignal is physics (Oliver–Soundararajan).

Any red → paste the failing section; we localise the bug.